<center>
    <h1>Generate Training set for <a style="color:red;">QM</a>Learn</h1>
    
    Pavanello Research Group
    
    August 2024
    
    MolSSI Workshop @ Buffalo
</center>

### `ASE` imports

In [1]:
from ase.optimize import LBFGS
import ase

### `QM`Learn imports

In [2]:
from qmlearn.api.api4ase import QMLCalculator
from qmlearn.drivers.mol import QMMol
from qmlearn.model import QMModel
from qmlearn.io import read_images
from qmlearn.preprocessing import AtomsCreater, build_train_atoms, build_properties

/Users/michele/Documents/Buffalo/qml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/michele/Documents/Buffalo/qml/lib/python3.10/site-packages/pyscf/dft/libxc.py:772: UserWarning: Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, the same to the B3LYP functional in Gaussian and ORCA (issue 1480). To restore the VWN5 definition, you can put the setting "B3LYP_WITH_VWN5 = True" in pyscf_conf.py
  warnings.warn('Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, '


### Build ref`QMmol` object

In [3]:
basis = '6-31g*'
xc = 'B3LYP, B3LYP'
method = 'rks'

from ase.build import molecule
atoms = molecule('H2O')

refqmmol = QMMol(atoms = atoms, method = method, basis = basis, xc = xc)

### Build `QM`Learn `ASE` calculator that uses `engine`

In [4]:
atoms.calc = QMLCalculator(qmmodel = refqmmol, method = 'engine')

#### This `QMLCalculator` with `method=engine` will run DFT, not ML. To run ML, use `method=gamma`. But first we need to train the ML models!

### Geometry optimization for ref`QMmol`

In [5]:
LBFGS(atoms).run(fmax = 1e-5)

       Step     Time          Energy         fmax
LBFGS:    0 11:05:11    -2341.399292       16.0312
LBFGS:    1 11:05:12    -2343.252263       15.0321
LBFGS:    2 11:05:12    -2344.074089        6.2177
LBFGS:    3 11:05:13    -2344.391187        2.4351
LBFGS:    4 11:05:13    -2344.464656        1.3837
LBFGS:    5 11:05:14    -2344.479228        0.4425
LBFGS:    6 11:05:14    -2344.480976        0.0431
LBFGS:    7 11:05:15    -2344.480992        0.0012
LBFGS:    8 11:05:15    -2344.480992        0.0000


True

### What's in ref`QMmol`?

In [6]:
refqmmol.init_kwargs

{'atoms': Atoms(symbols='OH2', pbc=False, calculator=QMLCalculator(...)),
 'engine_name': 'pyscf',
 'method': 'rks',
 'basis': '6-31g*',
 'xc': 'B3LYP, B3LYP',
 'occs': None,
 'refatoms': None,
 'engine_options': {},
 'charge': None,
 'engine': None,
 'stereo': True,
 'rotate_method': 'kabsch',
 'reorder_method': 'none',
 'use_reflection': True,
 'ignore_hydrogen': False}

### Clear `ir` folder to correctly initialize the `vib` object (`ASE` should fix this)

In [7]:
!rm -rf ./ir/

### Calculate by finite differences the dynamical matrix

In [8]:
# IR spectra
from ase.vibrations import Infrared
ir = Infrared(atoms, nfree = 4)
ir.run()

In [9]:
vib = ir.get_vibrations()

### Get vibrational modes and frequencies

In [10]:
modes = vib.get_modes()
frequencies = vib.get_frequencies().real

### Set temperature and seed for reproducibility

In [11]:
random_seed = 8888
charge = 0
temperature = 300

### `nsamples`$\propto N_{vib}^3$, and `tol` needs to be large enough (RMSD>`tol`)

In [12]:
nsamples = 27
tol = 3E-2

### Random points sampling normal mode displacements

In [13]:
creater = AtomsCreater(modes = modes, frequencies = frequencies, atoms = atoms, temperature = temperature, random_seed = random_seed)
images = build_train_atoms(creater, nsamples = nsamples, tol = tol, refatoms = atoms)

Start build
Get 27 samples at 152017 step.


### Let's visualize the training set

In [14]:
from ase.visualize import view
view(images)

<Popen: returncode: None args: ['/Users/michele/Documents/Buffalo/qml/bin/py...>

## Build training set of target properties ($\hat \gamma$, $\vec\nabla_R E$, $\vec\mu$)

### Select target properties and compute them

In [15]:
prop = ['vext', 'gamma', 'energy', 'forces', 'dipole', 'ke']
properties = build_properties(images, refqmmol = refqmmol, properties = prop)

100%|██████████████████████████████████████████████████████████████████████████████| 27/27 [00:15<00:00,  1.75it/s]


### Write database file (DB) in `.hdf5` format

In [16]:
from qmlearn.io import write_db
write_db('h2o_data.hdf5', refqmmol, images, properties)

Names in database:  ['rks', 'rks/qmmol', 'rks/train_atoms_27', 'rks/train_props_27']


## How do we use the DB file?

### Read DB

In [17]:
from qmlearn.io import read_db
rks_db = read_db('h2o_data.hdf5',names='rks')

Guess DB names : {'qmmol': 'rks/qmmol', 'atoms': 'rks/train_atoms_27', 'properties': 'rks/train_props_27'}


### Good habits:
 - make one DB for each molecule
 - write all models (levels of theory) in the DB file using `mode='a'` to append.
 - the `names` of the model is `QM`mol's `method` argument (see cell [3], will be improved in later `QM`Learn versions)

In [18]:
rks_db.keys()

dict_keys(['qmmol', 'atoms', 'properties', '_names'])

In [19]:
aqmmol = rks_db['qmmol']

In [20]:
aqmmol.atoms == atoms

True

In [21]:
rks_db['atoms'][0]== aqmmol.atoms

True